# 02 — Clean and Validate (Incremental Bronze → Silver)

ADF passes `batch_id`. Missing source folders are allowed: each run processes only the source folders present in that Bronze batch, then upserts Silver by URL.


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

dbutils.widgets.text("batch_id", "")
batch_id = dbutils.widgets.get("batch_id").strip()
if not batch_id:
    raise ValueError("batch_id is required")

BASE_PATH = "abfss://edtech@edtechpipline26.dfs.core.windows.net"
RAW_PATH = f"{BASE_PATH}/raw/batch_id={batch_id}"
VALIDATED_PATH = f"{BASE_PATH}/interim/validated"
REJECTED_PATH = f"{BASE_PATH}/interim/rejected"


In [0]:
# Discover which source folders exist in this batch.
# Incremental batches are allowed to contain only the sources that have new data.
try:
    available_sources = {
        item.name.rstrip("/")
        for item in dbutils.fs.ls(RAW_PATH)
        if item.isDir()
    }
except Exception as e:
    raise ValueError(f"Batch path does not exist or cannot be read: {RAW_PATH}") from e

supported_sources = {"devto", "freecodecamp", "geeksforgeeks", "medium", "pluralsight"}
available_sources = available_sources.intersection(supported_sources)

if not available_sources:
    raise ValueError(f"No supported source folders found in batch {batch_id}")

def standard_date(c):
    return F.to_date(F.expr(f"try_to_timestamp(nullif(trim(`{c}`), ''))"))

def pluralsight_date(c):
    return F.to_date(F.expr(f"try_to_timestamp(nullif(trim(`{c}`), ''), 'MMM d, yyyy')"))

standardized_dfs = []

if "devto" in available_sources:
    df = spark.read.option("multiLine", True).json(f"{RAW_PATH}/devto/*.json")
    tag_text = F.lower(F.concat_ws(" ", F.col("tag_list")))
    standardized_dfs.append(df.select(
        F.lit("dev.to").alias("source"),
        F.when(tag_text.rlike(r"(^|\W)(ai|machinelearning|deeplearning|llm|artificialintelligence)(\W|$)"),"AI")
         .when(tag_text.rlike(r"(^|\W)(cloud|aws|azure|gcp|devops|kubernetes)(\W|$)"),"Cloud")
         .otherwise("Data").alias("category"),
        F.col("title").cast("string").alias("title"),
        F.coalesce(F.col("user.name"),F.col("user.username")).cast("string").alias("author"),
        standard_date("published_at").alias("publication_date"),
        F.col("description").cast("string").alias("description"),
        F.col("url").cast("string").alias("url"),
        F.col("body_markdown").cast("string").alias("content"),
        F.concat_ws(", ",F.col("tag_list")).alias("tags")))

if "freecodecamp" in available_sources:
    df=(spark.read.option("header",True).option("inferSchema",True).option("multiLine",True)
        .option("quote",'"').option("escape",'"').csv(f"{RAW_PATH}/freecodecamp/*.csv"))
    standardized_dfs.append(df.select(
        F.lit("freeCodeCamp").alias("source"),F.coalesce(F.col("topic"),F.col("category")).cast("string").alias("category"),
        F.col("title").cast("string").alias("title"),F.col("author").cast("string").alias("author"),
        standard_date("publication_date").alias("publication_date"),F.col("description").cast("string").alias("description"),
        F.col("url").cast("string").alias("url"),F.col("content").cast("string").alias("content"),
        F.col("matched_keywords").cast("string").alias("tags")))

for folder, source_name in [("geeksforgeeks","GeeksforGeeks"),("medium","Medium"),("pluralsight","Pluralsight")]:
    if folder in available_sources:
        df=spark.read.option("multiLine",True).json(f"{RAW_PATH}/{folder}/*.json")
        date_col = pluralsight_date("publication_date") if folder=="pluralsight" else standard_date("publication_date")
        standardized_dfs.append(df.select(
            F.lit(source_name).alias("source"),F.col("category").cast("string").alias("category"),
            F.col("title").cast("string").alias("title"),F.col("author").cast("string").alias("author"),
            date_col.alias("publication_date"),F.col("description").cast("string").alias("description"),
            F.col("url").cast("string").alias("url"),F.col("content").cast("string").alias("content"),
            F.col("tags").cast("string").alias("tags")))

combined_df = standardized_dfs[0]
for df in standardized_dfs[1:]:
    combined_df = combined_df.unionByName(df)


In [0]:
k = F.lower(F.trim(F.col("category")))

cleaned_df = (combined_df
    .withColumn("category",
        F.when(k.isin("ai","artificial intelligence","ai & data","ai and data"),"AI")
         .when(k.isin("data","data science","datascience"),"Data")
         .when(k.isin("cloud","cloud computing"),"Cloud")
         .otherwise(F.col("category")))
    .withColumn("batch_id",F.lit(batch_id))
    .withColumn("ingested_at",F.current_timestamp()))

missing_title = F.col("title").isNull() | (F.trim(F.col("title"))=="")
missing_url = F.col("url").isNull() | (F.trim(F.col("url"))=="")
missing_content = F.col("content").isNull() | (F.trim(F.col("content"))=="")
invalid = missing_title | missing_url | missing_content

validated_batch = cleaned_df.filter(~invalid).dropDuplicates(["url"])

rejected_batch = cleaned_df.filter(invalid).withColumn(
    "rejection_reason",F.concat_ws("; ",
        F.when(missing_title,F.lit("missing title")),
        F.when(missing_url,F.lit("missing url")),
        F.when(missing_content,F.lit("missing content"))))

In [0]:
if DeltaTable.isDeltaTable(spark, VALIDATED_PATH):
    (DeltaTable.forPath(spark, VALIDATED_PATH).alias("t")
     .merge(validated_batch.alias("s"), "t.url = s.url")
     .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute())
else:
    validated_batch.write.format("delta").mode("overwrite").save(VALIDATED_PATH)

# Rejected rows are retained as an audit trail and protected against same-batch retries.
if DeltaTable.isDeltaTable(spark, REJECTED_PATH):
    (DeltaTable.forPath(spark, REJECTED_PATH).alias("t")
     .merge(rejected_batch.alias("s"),
            "t.batch_id = s.batch_id AND t.url <=> s.url AND t.title <=> s.title")
     .whenNotMatchedInsertAll().execute())
else:
    rejected_batch.write.format("delta").mode("overwrite").save(REJECTED_PATH)
